In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("Superstore.csv", encoding='latin1')

print("Initial Data Shape:", df.shape)
display(df.head())

Initial Data Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
print("\nMissing Values per column:\n", df.isnull().sum())


Missing Values per column:
 Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [5]:
initial_rows = len(df)
df = df.drop_duplicates()
print(f"\nRemoved {initial_rows - len(df)} duplicate rows.")


Removed 0 duplicate rows.


In [6]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

In [28]:
df['Order_year'] = df['Order Date'].dt.year
df['Order_Month'] = df['Order Date'].dt.month
df['Profit_Margin'] = (df['Profit'] / df['Sales']) * 100

In [31]:
df.to_csv('Cleaned_Superstore_Data.csv', index=False)
print("\nData Cleaning Complete! Cleaned file saved as 'Cleaned_Superstore_data.csv'.")


Data Cleaning Complete! Cleaned file saved as 'Cleaned_Superstore_data.csv'.


In [32]:
import sqlite3
conn = sqlite3.connect('ecommerce_project.db')
cursor = conn.cursor()

df.to_sql('cleaned_superstore_data', conn, if_exists='replace', index=False)

print("DataFrame successfully loaded into SQLite dataBase table: 'cleaned_superstore_data'")

DataFrame successfully loaded into SQLite dataBase table: 'cleaned_superstore_data'


In [33]:
query_1 = """
SELECT
  Region,
  ROUND(SUM(Sales), 2) AS Total_Revenue,
  ROUND(SUM(Profit), 2) AS Total_Profit
FROM
  cleaned_superstore_data
GROUP BY
  Region
ORDER BY
  Total_Revenue DESC;
"""

df_result_1 = pd.read_sql_query(query_1, conn)
print("--- 1. Region-wise Sales & Profit ---")
display(df_result_1)

--- 1. Region-wise Sales & Profit ---


,Region,Total_Revenue,Total_Profit
0,West,725457.82,108418.45
1,East,678781.24,91522.78
2,Central,501239.89,39706.36
3,South,391721.91,46749.43


In [34]:
query_2 = """
SELECT
    "Product Name",
    ROUND(SUM(Sales), 2) AS Total_Sales,
    SUM(Quantity) AS Total_Quantity_Sold
FROM
    cleaned_superstore_data
GROUP BY
    "Product Name"
ORDER BY
    Total_Sales DESC
LIMIT 5;
"""

df_result_2 = pd.read_sql_query(query_2, conn)
print("---2. Top 5 best Selling Products ---")
display(df_result_2)

---2. Top 5 best Selling Products ---


,Product Name,Total_Sales,Total_Quantity_Sold
0,Canon imageCLASS 2200 Advanced Copier,61599.82,20
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,31
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,6
3,HON 5400 Series Task Chairs for Big and Tall,21870.58,39
4,GBC DocuBind TL300 Electric Binding System,19823.48,37


In [35]:
query_3 = """
SELECT
  Order_Year,
  Order_Month,
  ROUND(SUM(Sales), 2) AS Monthly_Revenue
FROM
  cleaned_superstore_data
GROUP BY
  Order_Year,
  Order_Month
ORDER BY
  Order_Year,
  Order_Month;
"""

df_result_3 = pd.read_sql_query(query_3, conn)
print("--- 3. Monthly sales Trend ---")
display(df_result_3)

--- 3. Monthly sales Trend ---


,Order_year,Order_Month,Monthly_Revenue
0,2014,1,14236.90
1,2014,2,4519.89
2,2014,3,55691.01
3,2014,4,28295.35
4,2014,5,23648.29
5,2014,6,34595.13
6,2014,7,33946.39
7,2014,8,27909.47
8,2014,9,81777.35
9,2014,10,31453.39
